In [1]:
# pip install s3fs fsspec zarr numpy

import s3fs, fsspec, zarr, numpy as np

# SDOML (AWS Open Data) bucket + prefix (public; no credentials needed)
BUCKET = "gov-nasa-hdrl-data1"
PREFIX = "contrib/fdl-sdoml/"   # region: us-west-2

In [2]:
fs = s3fs.S3FileSystem(anon=True)
root = f"{BUCKET}/{PREFIX}".rstrip("/")

In [3]:
# 1) Find candidate AIA 193Å Zarr stores
candidates = []
for p in fs.find(root):
    lp = p.lower()
    if ".zarr" in lp and "aia" in lp and ("193" in lp or "193a" in lp or "19.3" in lp):
        candidates.append(p)

print("Found", len(candidates), "candidates (showing up to 5):")
for c in candidates[:5]:
    print(" -", c)

Found 0 candidates (showing up to 5):


In [ ]:
assert candidates, "No AIA 193Å Zarr stores found — adjust PREFIX or search logic."
store_path = "s3://" + candidates[0]   # pick one

# 2) Open Zarr lazily via fsspec
mapper = fsspec.get_mapper(store_path, anon=True)
z = zarr.open(mapper, mode="r")

# Optional: inspect keys/shape
try:
    print("Arrays:", list(z.array_keys())[:10])
except Exception:
    pass

# 3) Read a single frame (supports [T,H,W] or [H,W,T])
arr = np.asarray(z)
if arr.ndim == 3:
    if arr.shape[0] > 10:       # [T,H,W]
        frame0 = np.array(z[0, :, :], dtype=np.float32)
    else:                        # [H,W,T]
        frame0 = np.array(z[:, :, 0], dtype=np.float32)
elif arr.ndim == 2:
    frame0 = np.array(z[:], dtype=np.float32)
else:
    raise ValueError(f"Unexpected shape: {arr.shape}")

print("Frame shape:", frame0.shape, "dtype:", frame0.dtype, "min/max:", float(frame0.min()), float(frame0.max()))